# Feature attributions for Integrated Gradients

Calculate and plot IG attributions on LAAT and CAML and find standard deviations for random attributions

In [ ]:
from captum.attr import configure_interpretable_embedding_layer
from captum.attr import remove_interpretable_embedding_layer
from matplotlib import pyplot as plt
import torch
import numpy as np
import random

import config
import loader
import attributor

SCOPES = config.SCOPES
SUB_BITS = config.SUB_BITS
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
METHOD_NAME = 'ig'
N_STEPS = 200

## LAAT

In [ ]:
# Load model
model_name = 'laat'
model, dataloader = loader.load_model_and_data(model_name)
def model_wrapper(*args, **kwargs):
    output, _ = model(*args, **kwargs)
    return torch.sigmoid(output[1])
int_emb = configure_interpretable_embedding_layer(model, 'embedding')
model.train()

In [ ]:
# Create IG attribution methods
methods = {}
for scope in SCOPES:
    methods[scope] = attributor.create_method(model_wrapper, scope, METHOD_NAME)

In [ ]:
predss_laat = {}
attrs_laat = {}
N = 3
for idx, tup in enumerate(dataloader):
    # Attribute only the N first samples of the subset
    if SUB_BITS[idx] == 0 or np.count_nonzero(SUB_BITS[:idx]) > N-1:
        continue
    print("Attributing sample", idx)
    predss_laat[idx] = {}
    attrs_laat[idx] = {}
    input_embed, base_embed, afa = loader.prepare_input(model_name, tup, int_emb)
    preds = model_wrapper(input_embed, afa)[0]
    for target_idx, pred in enumerate(preds):
        if pred.item() > config.THRESH:
            # Attribute only 50% to save time
            if random.uniform(0, 1) > 0.5:
                continue
            print("Attributing target_idx", target_idx)
            predss_laat[idx][target_idx] = pred.item()
            attrs_laat[idx][target_idx] = {}
            # Compute attributions
            for scope in SCOPES:
                a = attributor.attribute(scope,
                                         METHOD_NAME,
                                         methods[scope],
                                         input_embed,
                                         base_embed,
                                         afa,
                                         target_idx,
                                         n_steps=N_STEPS,
                                         n_samples=None)
                attrs_laat[idx][target_idx][scope] = a

In [ ]:
# Find standard deviation for local and global random attributions
# Take std of IG on first sample, first target
for idx, preds in predss_laat.items():
    for target_idx, pred in preds.items():
        std_local = torch.std(attrs_laat[idx][target_idx]['local'], unbiased=False).item()
        std_global = torch.std(attrs_laat[idx][target_idx]['global'], unbiased=False).item()
        break
    break
std_local = round(std_local, 6)
std_global = round(std_global, 6)
print("std_local:", std_local)
print("std_global:", std_global)

In [ ]:
# Plot attributions
ylim = 0.05
for idx, preds in predss_laat.items():
    for target_idx, pred in preds.items():
        print("sample_idx:", idx)
        print("target_idx:", target_idx)
        print("pred:", round(pred, 2))
        fig, axs = plt.subplots(1, 2, figsize=(14,6))
        idx_local, idx_global = 0, 1
        a_local = attrs_laat[idx][target_idx]['local']
        a_local = a_local.sum(dim=2).squeeze(0).cpu().detach().numpy()
        a_global = attrs_laat[idx][target_idx]['global']
        a_global = a_global.sum(dim=2).squeeze(0).cpu().detach().numpy()
        axs[idx_local].bar(range(len(a_local)), a_local, facecolor='b', alpha=0.75)
        axs[idx_local].set_ylabel(f"local attribution values", fontsize=20)
        axs[idx_local].set_xlabel(f"words in document", fontsize=20)
        axs[idx_local].set_ylim(-ylim, ylim)
        axs[idx_global].bar(range(len(a_global)), a_global, facecolor='b', alpha=0.75)
        axs[idx_global].set_ylabel(f"global attribution values", fontsize=20)
        axs[idx_global].set_xlabel(f"words in document", fontsize=20)
        axs[idx_global].set_ylim(-ylim, ylim)
        fig.tight_layout()
        fig.show()
        basename = f"plots/distributions_ig/laat_ig{N_STEPS}_sample{idx}_label{target_idx}"
        fig.savefig(f"{basename}.png", format="png")
        fig.savefig(f"{basename}.eps", format="eps")

In [ ]:
model.train(mode=False)
remove_interpretable_embedding_layer(model, int_emb)

## CAML

In [ ]:
# Load model
model_name = 'caml'
model, dataloader = loader.load_model_and_data(model_name)
def model_wrapper(*args, **kwargs):
    output, _, _ = model(*args, **kwargs)
    return torch.sigmoid(output)
int_emb = configure_interpretable_embedding_layer(model, 'embed')
model.train()

In [ ]:
# Create IG attribution methods
methods = {}
for scope in SCOPES:
    methods[scope] = attributor.create_method(model_wrapper, scope, METHOD_NAME)

In [ ]:
predss_caml = {}
attrs_caml = {}
N = 3
for idx, tup in enumerate(dataloader):
    # Attribute only the N first samples of the subset
    if SUB_BITS[idx] == 0 or np.count_nonzero(SUB_BITS[:idx]) > N-1:
        continue
    print("Attributing sample", idx)
    predss_caml[idx] = {}
    attrs_caml[idx] = {}
    input_embed, base_embed, afa = loader.prepare_input(model_name, tup, int_emb)
    preds = model_wrapper(input_embed, afa)[0]
    for target_idx, pred in enumerate(preds):
        if pred.item() > config.THRESH:
            # Attribute only 90% to save time
            if random.uniform(0, 1) > 0.9:
                continue
            print("Attributing target_idx", target_idx)
            predss_caml[idx][target_idx] = pred.item()
            attrs_caml[idx][target_idx] = {}
            # Compute attributions
            for scope in config.SCOPES:
                a = attributor.attribute(scope,
                                         METHOD_NAME,
                                         methods[scope],
                                         input_embed,
                                         base_embed,
                                         afa,
                                         target_idx,
                                         n_steps=N_STEPS,
                                         n_samples=None)
                attrs_caml[idx][target_idx][scope] = a

In [ ]:
# Find standard deviation for local and global random attributions
# Take std of IG on first sample, first target
for idx, preds in predss_caml.items():
    for target_idx, pred in preds.items():
        std_local = torch.std(attrs_caml[idx][target_idx]['local'], unbiased=False).item()
        std_global = torch.std(attrs_caml[idx][target_idx]['global'], unbiased=False).item()
        break
    break
std_local = round(std_local, 6)
std_global = round(std_global, 6)
print("std_local:", std_local)
print("std_global:", std_global)

In [ ]:
# Plot attributions
ylim = 0.4
for idx, preds in predss_caml.items():
    for target_idx, pred in preds.items():
        print("sample_idx:", idx)
        print("target_idx:", target_idx)
        print("pred:", round(pred, 2))
        fig, axs = plt.subplots(1, 2, figsize=(14,6))
        idx_local, idx_global = 0, 1
        a_local = attrs_caml[idx][target_idx]['local']
        a_local = a_local.sum(dim=2).squeeze(0).cpu().detach().numpy()
        a_global = attrs_caml[idx][target_idx]['global']
        a_global = a_global.sum(dim=2).squeeze(0).cpu().detach().numpy()
        axs[idx_local].bar(range(len(a_local)), a_local, facecolor='b', alpha=0.75)
        axs[idx_local].set_ylabel(f"local attribution values", fontsize=20)
        axs[idx_local].set_xlabel(f"words in document", fontsize=20)
        axs[idx_local].set_ylim(-ylim, ylim)
        axs[idx_global].bar(range(len(a_global)), a_global, facecolor='b', alpha=0.75)
        axs[idx_global].set_ylabel(f"global attribution values", fontsize=20)
        axs[idx_global].set_xlabel(f"words in document", fontsize=20)
        axs[idx_global].set_ylim(-ylim, ylim)
        fig.tight_layout()
        fig.show()
        basename = f"plots/distributions_ig/caml_ig{N_STEPS}_sample{idx}_label{target_idx}"
        fig.savefig(f"{basename}.png", format="png")
        fig.savefig(f"{basename}.eps", format="eps")

In [ ]:
model.train(mode=False)
remove_interpretable_embedding_layer(model, int_emb)